# StormEngine V8 — Processor family development

This experiment isolates temporal forecasting skill: dense ERA5 grids from the past 12 hours are used to predict the next 6 hours. It does **not** load DPC, Open-Meteo, sparse stations, Encoder weights, Decoder weights, or 2017 test data.

Both candidates use 2013–2015 for training, 2016 for validation, a 3-hour development stride, the same sea-weighted loss, and validation early stopping.

In [ ]:
from pathlib import Path
import json, subprocess, sys, torch
here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').is_file(), REPO
DEVICE = 'cuda'
CONFIGS = {
    'ConvGRU': REPO / 'configs' / 'v8_processor_dev3y_convgru.yaml',
    'Factorized-ViT': REPO / 'configs' / 'v8_processor_dev3y_vit.yaml',
}
OUTPUTS = {
    'ConvGRU': REPO / 'artifacts' / 'v8_processor_dev3y_convgru_seed42',
    'Factorized-ViT': REPO / 'artifacts' / 'v8_processor_dev3y_vit_seed42',
}
print('Repository:', REPO)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CUDA unavailable')

## 1. Shape, backward, and peak-memory preflight

Run both preflights sequentially. The estimate below is deliberately conservative because two CUDA processes also create separate CUDA contexts.

In [ ]:
def preflight(config):
    command = [sys.executable, '-u', str(REPO / 'scripts' / 'train_dense_processor.py'),
               'preflight', '--device', DEVICE, '--config', str(config)]
    completed = subprocess.run(command, cwd=REPO, check=True, text=True, capture_output=True)
    text = completed.stdout
    result = json.loads(text[text.index('{'):])
    print(result['family'], 'shape=', result['prediction'],
          'parameters=', f"{result['contract']['trainable_parameters']:,}",
          'peak=', f"{result['peak_cuda_allocated_gib']:.2f} GiB")
    return result
profiles = {name: preflight(config) for name, config in CONFIGS.items()}
free_bytes, total_bytes = torch.cuda.mem_get_info()
estimated = 1.35 * sum(item['peak_cuda_allocated_bytes'] for item in profiles.values()) + 1.5 * 1024**3
PARALLEL_MEMORY_SAFE = estimated < free_bytes
print(f'GPU free/total: {free_bytes/1024**3:.2f}/{total_bytes/1024**3:.2f} GiB')
print(f'Conservative parallel requirement: {estimated/1024**3:.2f} GiB')
print('Parallel memory check:', 'PASS' if PARALLEL_MEMORY_SAFE else 'FAIL — run sequentially')

## 2. Smoke checks

These are pipeline checks only and are stored separately from the development runs.

In [ ]:
for name, config in CONFIGS.items():
    smoke_output = OUTPUTS[name].with_name(OUTPUTS[name].name + '_smoke')
    summary = smoke_output / 'smoke_summary.json'
    if summary.is_file():
        print('Already complete:', name, summary)
        continue
    command = [sys.executable, '-u', str(REPO / 'scripts' / 'train_dense_processor.py'),
               'smoke', '--device', DEVICE, '--config', str(config),
               '--output-dir', str(smoke_output)]
    subprocess.run(command, cwd=REPO, check=True)

## 3. Launch both converged development runs

On Windows this opens two independent console windows. Set `RUN_BOTH=True` only when the memory preflight passes. GPU compute is still shared, so parallel execution may not be faster than sequential execution. Each run supports automatic resume from its own `last.pt`.

In [ ]:
RUN_BOTH = False
processes = {}
if RUN_BOTH:
    assert sys.platform == 'win32', 'Parallel launcher is intended for the Windows CUDA computer.'
    assert PARALLEL_MEMORY_SAFE, 'Peak-memory check failed; run the two commands sequentially.'
    for name, config in CONFIGS.items():
        summary = OUTPUTS[name] / 'develop_summary.json'
        if summary.is_file():
            print('Already complete:', name, summary)
            continue
        command = [sys.executable, '-u', str(REPO / 'scripts' / 'train_dense_processor.py'),
                   'develop', '--device', DEVICE, '--config', str(config)]
        checkpoint = OUTPUTS[name] / 'last.pt'
        if checkpoint.is_file():
            command += ['--resume', str(checkpoint)]
        processes[name] = subprocess.Popen(command, cwd=REPO,
                                             creationflags=subprocess.CREATE_NEW_CONSOLE)
        print('Launched:', name, 'PID=', processes[name].pid)
else:
    print('Set RUN_BOTH=True after the preflight and smoke checks pass.')

## 4. Compare only after both runs converge

This remains a single-seed family screen. Do not choose the final Processor until the top comparison has been repeated with a second seed.

In [ ]:
summaries = [OUTPUTS[name] / 'develop_summary.json' for name in OUTPUTS]
if all(path.is_file() for path in summaries):
    subprocess.run([sys.executable, '-u', str(REPO / 'scripts' / 'compare_dense_processors.py'),
                    *map(str, summaries), '--output',
                    str(REPO / 'artifacts' / 'v8_processor_dev3y_family_comparison_seed42.json')],
                   cwd=REPO, check=True)
else:
    print('Wait until both develop_summary.json files exist.')